# DigiGestures: Gesture Classifier Training Pipeline

This notebook contains the complete self-contained pipeline to load collected hand landmark features, train a Random Forest classifier, visualize its performance, and export the trained model package.

In [ ]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split

# Setup Paths relative to the notebooks folder
PROJECT_ROOT = Path("..").resolve()
DATA_PATH = PROJECT_ROOT / "data" / "hand_data.csv"
MODEL_PATH = PROJECT_ROOT / "models" / "gesture_model.pkl"

LETTERS = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
DIGITS = "123456789"
LABEL_TO_ID = {label: index for index, label in enumerate(LETTERS + DIGITS)}
ID_TO_LABEL = {index: label for label, index in LABEL_TO_ID.items()}
TOTAL_FEATURES = 126

## 1. Define Helper Functions

If your `hand_data.csv` is empty or doesn't exist yet, `load_dataset` will automatically generate a synthetic dataset representing a few gestures (`A`, `B`, `C`, `1`, `2`, `3`) so you can run the notebook and test the classifier immediately.

In [ ]:
def generate_synthetic_data(num_samples_per_class: int = 50) -> pd.DataFrame:
    """Generate synthetic hand landmark data for pipeline verification."""
    print("Generating synthetic hand data for verification...")
    np.random.seed(42)
    
    target_labels = ["A", "B", "C", "1", "2", "3"]
    target_ids = [LABEL_TO_ID[label] for label in target_labels]
    records = []
    
    for label_id in target_ids:
        for _ in range(num_samples_per_class):
            left_hand = np.zeros(63)
            right_hand = np.zeros(63)
            base_hand = []
            
            for finger_idx in range(5):
                angle = finger_idx * 0.4
                for joint_idx in range(1, 5):
                    is_open = True
                    if label_id == LABEL_TO_ID["A"]:
                        is_open = False
                    elif label_id == LABEL_TO_ID["1"] and finger_idx != 1:
                        is_open = False
                    elif label_id == LABEL_TO_ID["2"] and finger_idx not in (1, 2):
                        is_open = False
                    elif label_id == LABEL_TO_ID["3"] and finger_idx not in (1, 2, 3):
                        is_open = False
                    
                    extension = (joint_idx * 0.1) if is_open else (joint_idx * 0.02)
                    base_hand.extend([
                        np.cos(angle) * extension + np.random.normal(0, 0.01),
                        np.sin(angle) * extension + np.random.normal(0, 0.01),
                        -joint_idx * 0.02 + np.random.normal(0, 0.01)
                    ])
            
            right_hand_landmarks = [0.0, 0.0, 0.0] + base_hand
            features = left_hand.tolist() + right_hand_landmarks
            features.append(label_id)
            records.append(features)
            
    columns = []
    for hand_name in ("left", "right"):
        for landmark_index in range(21):
            for axis in ("x", "y", "z"):
                columns.append(f"{hand_name}_lm{landmark_index}_{axis}")
    columns.append("label")
    
    return pd.DataFrame(records, columns=columns)

def load_dataset(csv_path: Path) -> pd.DataFrame:
    """Load collected data or fallback to synthetic data if empty."""
    if not csv_path.exists() or csv_path.stat().st_size <= 2000:
        df = generate_synthetic_data()
        csv_path.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(csv_path, index=False)
        print(f"Saved synthetic dataset to {csv_path}")
        return df
        
    df = pd.read_csv(csv_path)
    if len(df) < 10:
        print(f"Dataset has only {len(df)} samples. Adding synthetic data to help training...")
        synthetic_df = generate_synthetic_data()
        df = pd.concat([df, synthetic_df], ignore_index=True)
        
    return df

## 2. Load Dataset

In [ ]:
df = load_dataset(DATA_PATH)
print(f"Dataset shape: {df.shape}")
print(f"Target class counts:\n{df['label'].map(ID_TO_LABEL).value_counts()}")

## 3. Train / Test Split

In [ ]:
X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Testing set: {X_test.shape[0]} samples")

## 4. Train Classifier

In [ ]:
print("Training Random Forest Classifier...")
clf = RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42)
clf.fit(X_train, y_train)
print("Model training complete.")

## 5. Evaluate Performance

In [ ]:
y_pred = clf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {accuracy * 100:.2f}%\n")

unique_labels = np.unique(np.concatenate([y_test, y_pred]))
target_names = [ID_TO_LABEL[lid] for lid in unique_labels]

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=target_names))

# Plot confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm, 
    annot=True, 
    fmt="d", 
    cmap="Blues", 
    xticklabels=target_names, 
    yticklabels=target_names
)
plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()

## 6. Save Model Package

In [ ]:
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)

model_package = {
    "classifier": clf,
    "label_to_id": LABEL_TO_ID,
    "id_to_label": ID_TO_LABEL,
    "feature_count": TOTAL_FEATURES
}

with MODEL_PATH.open("wb") as file:
    pickle.dump(model_package, file)
    
print(f"Model package successfully saved to {MODEL_PATH}")